# OHAYO - Adapter Serving

Private inference trial, not training. Select GPU T4 x2 and Internet; attach the existing NF4 checkpoint and the new private serving Input. Enable NGROK_AUTHTOKEN and OHAYO_INFERENCE_TOKEN in Secrets.

Run cells 1-4. Start in a fresh kernel. Keep the old Base notebook/version. For rollback restart Python, change ARM to base_rag in cell 1, then run cells 1-4 again. Never Run All in the old training notebook.


In [ ]:
from pathlib import Path
import hashlib, json, os, zipfile

ARM = "tuned_rag"  # Rollback: restart Python, set "base_rag", then run cells 1-4.
assert ARM in ("base_rag", "tuned_rag")
if globals().get('_ohayo_service'):
    raise RuntimeError('Restart & Clear Cell Outputs before changing arm or reimporting this notebook.')
EXPECTED_PACKAGE_SHA256 = '4485b08d9aef43331a8b5aa60b0bc16966295cc95de3b086c01080a8384348d8'
EXPECTED_MANIFEST_SHA256 = 'ecc0a5256e92a0585c0ab066ee6bb5dba9ffd5456be1f80f9f8903b56cadd621'
EXPECTED_ADAPTER_SHA256 = '51713be1e630135099f95ed08c89b5be8669b75c9e1908a1346c5c9cc4942ae7'
candidates = [p for p in Path('/kaggle/input').rglob('serving-input.zip')
              if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_PACKAGE_SHA256]
SERVING_PACKAGE_SHA256 = EXPECTED_PACKAGE_SHA256
SERVING_PACKAGE = Path('/kaggle/working/ayo-serving-' + EXPECTED_PACKAGE_SHA256[:12])
if candidates:
    with zipfile.ZipFile(candidates[0]) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or sum(i.file_size for i in archive.infolist()) > 160 * 1024**2:
            raise RuntimeError('Invalid or oversized serving ZIP')
        payloads = {name:archive.read(name) for name in names}
else:
    manifests = [p for p in Path('/kaggle/input').rglob('manifest.json')
                 if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256]
    if not manifests:
        raise RuntimeError('Add the matching private serving Input and keep the NF4 checkpoint Input too.')
    source_root = manifests[0].parent.resolve()
    inventory = json.loads(manifests[0].read_bytes())['files']
    if sum(info['size_bytes'] for info in inventory.values()) > 160 * 1024**2:
        raise RuntimeError('Oversized extracted Input')
    payloads = {'manifest.json':manifests[0].read_bytes()}
    for name in inventory:
        path = source_root / name
        if path.is_symlink() or not path.resolve().is_relative_to(source_root):
            raise RuntimeError('Unsafe extracted Input path')
        payloads[name] = path.read_bytes()
if hashlib.sha256(payloads['manifest.json']).hexdigest() != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Serving manifest mismatch')
manifest = json.loads(payloads['manifest.json'])
if set(payloads) != set(manifest['files']) | {'manifest.json'}:
    raise RuntimeError('Unexpected package members')
for name, info in manifest['files'].items():
    relative = Path(name)
    if relative.is_absolute() or '..' in relative.parts or '\\' in name or ':' in name:
        raise RuntimeError('Unsafe package path')
    data = payloads[name]
    if len(data) != info['size_bytes'] or hashlib.sha256(data).hexdigest() != info['sha256']:
        raise RuntimeError('Package checksum mismatch')
    destination = SERVING_PACKAGE / relative
    if not destination.resolve().is_relative_to(SERVING_PACKAGE.resolve()) or destination.is_symlink():
        raise RuntimeError('Unsafe extraction destination')
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.read_bytes() != data:
        raise RuntimeError('Existing package changed; do not overwrite')
    destination.write_bytes(data)
del payloads
OHAYO_SERVING_ARM = ARM
os.environ.update(TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')
print('INPUT_VERIFIED:', ARM, '| no training dataset, no base weights in this ZIP')


In [ ]:
exec(compile((SERVING_PACKAGE / 'startup.py').read_text(encoding='utf-8'), 'ayo-serving-startup', 'exec'), globals())
print('SERVING_STARTED:', ARM, '| run the next cell to verify actual adapter state')


In [ ]:
import importlib.metadata, time
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from kaggle_secrets import UserSecretsClient

evidence = Path('/kaggle/working/ayo-serving-evidence-' + ARM)
evidence.mkdir(exist_ok=True)
report = dict(status='SERVING_VERIFICATION_FAILED',arm=ARM,
    package_sha256=EXPECTED_PACKAGE_SHA256,health=None,smoke=[],
    acceptance='Serving identity and synthetic gateway smoke only; not 36-case quality/security acceptance')
token = UserSecretsClient().get_secret('OHAYO_INFERENCE_TOKEN')
def local_request(path, data=None):
    body = json.dumps(data).encode() if data is not None else None
    request = Request('http://127.0.0.1:8765' + path, data=body,
        headers={'Authorization':'Bearer ' + token, 'Content-Type':'application/json'})
    try:
        with urlopen(request, timeout=110) as response:
            return response.status, json.load(response)
    except HTTPError as error:
        return error.code, json.loads(error.read(4096))

source = dict(source_id='S1',chunk_id='serving-smoke-1',
    document_id='11111111-1111-4111-8111-111111111111',
    version_id='22222222-2222-4222-8222-222222222222',
    document_title='Synthetic serving smoke',page_number=1,
    text='Chu kỳ lệnh gồm ba bước: nạp lệnh (Fetch), giải mã (Decode), thực thi (Execute).')
try:
    health_http, health = local_request('/health')
    report.update(health=health,health_http=health_http)
    assert health_http == 200 and health['ready'] and health['replicas'] == 2
    assert health['serving']['arm'] == ARM
    required = {'chat_history_v1','staged_hint_v1','diagnosed_hint_v1','summary_json_quotes_v1'}
    assert required <= set(health['capabilities'])
    assert len(health['serving']['replicas']) == 2
    for replica in health['serving']['replicas']:
        assert replica['adapter_sha256'] == (EXPECTED_ADAPTER_SHA256 if ARM == 'tuned_rag' else None)
        assert replica['tokenizer_source'] == 'verified_adapter_training_tokenizer'
        assert replica['tokenizer_json_sha256'] == manifest['files']['adapter/tokenizer/tokenizer.json']['sha256']
        if ARM == 'tuned_rag':
            assert replica['tensor_values_match'] and replica['loaded_adapter_tensors'] == 392
            assert replica['trainable_parameters'] == 0 and replica['active_adapter'] == 'default'
    report['dependencies'] = {name:importlib.metadata.version(name) for name in
        ['torch','transformers','tokenizers','bitsandbytes','accelerate','safetensors']}
    for attempt in range(2):
        started = time.monotonic()
        status, result = local_request('/generate', dict(question='Theo học liệu, chu kỳ lệnh gồm những bước nào?',
            mode='qa',contexts=[source],exercise=None,config=health['config'],history=[],history_protocol='chat_history_v1'))
        report['smoke'].append(dict(http=status,seconds=round(time.monotonic()-started,3),result=result))
    report['status'] = ('SERVING_SMOKE_PASSED' if all(row['http']==200
        and row['result'].get('output',{}).get('status')=='ANSWERED'
        and row['result']['output'].get('references')==[{'label':1,'source_id':'S1'}]
        for row in report['smoke']) else 'SERVING_SMOKE_FAILED')
except Exception as error:
    report['failure_type'] = type(error).__name__
(evidence/'verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(dict(status=report['status'],arm=ARM,serving=(report.get('health') or {}).get('serving'),
    smoke_http=[row['http'] for row in report['smoke']]),indent=2))
print('Run cell 4 even if smoke failed. Do not change the backend URL yet.')


In [ ]:
from datetime import datetime, timezone
from IPython.display import FileLink, display
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zip_path = Path('/kaggle/working/serving-' + ARM + '-' + stamp + '.zip')
with zipfile.ZipFile(zip_path,'x',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(evidence.glob('*.json')):
        archive.write(path,path.name)
receipt = dict(sha256=hashlib.sha256(zip_path.read_bytes()).hexdigest(),
    size_bytes=zip_path.stat().st_size,arm=ARM,package_sha256=EXPECTED_PACKAGE_SHA256)
receipt_path = zip_path.with_suffix('.receipt.json')
receipt_path.write_text(json.dumps(receipt,indent=2),encoding='utf-8')
display(FileLink(str(zip_path)),FileLink(str(receipt_path)))
print('Send ZIP + receipt and the cell 3 status. No training was performed.')
